# 08_silver_bathymetry_features.ipynb — Batimetría Bronze → Silver

Este notebook procesa las fuentes batimétricas:

```text
data/GEBCO Bathymetry/
data/EMODnet Bathymetry/
```

y genera:

```text
silver/bathymetry_features/bathymetry_features.parquet
```

Tabla objetivo, una fila por `zona_id`:

```text
zona_id, depth_100m, depth_500m, depth_1km, depth_2km,
mean_depth_1km, slope_0_500m, slope_500m_2km,
distance_to_10m_isobath, distance_to_20m_isobath,
bathymetry_roughness
```

Notas:
- Se prioriza NetCDF si está disponible.
- Se recorta a Canarias + margen para no cargar datos globales completos.
- La profundidad se estandariza como **metros positivos bajo el nivel del mar**.
- Las features se calculan por zona usando un punto offshore aproximado derivado de `orientacion_costa`.
- Si alguna zona no puede calcular una feature, se conserva `NaN` y se marca con flag.

## Celda 0 — Montar Google Drive

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## Celda 1 — Instalar librerías necesarias

In [2]:
!pip -q install xarray netCDF4 h5netcdf rioxarray rasterio pyproj geopandas pyarrow shapely fiona tqdm scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 65.3 MB/s eta 0:00:00


## Celda 2 — Imports, rutas y configuración

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd
import xarray as xr
import rioxarray
import rasterio
import pyarrow as pa
import pyarrow.parquet as pq
import re
import unicodedata
import json
import zipfile
import shutil
import gc
from tqdm.auto import tqdm
from pyproj import Geod

BASE_DIR = Path("/content/drive/MyDrive/AI Projects/DeepWave Canarias")
BRONZE_DIR = BASE_DIR / "data/bronze"
SILVER_DIR = BASE_DIR / "silver"

GEBCO_DIR = BRONZE_DIR / "GEBCO Bathymetry"
EMODNET_DIR = BRONZE_DIR / "EMODnet Bathymetry"
DIM_ZONE_PATH = SILVER_DIR / "beach_geography" / "beach_geography.parquet"

OUT_DIR = SILVER_DIR / "bathymetry_features"
QC_DIR = SILVER_DIR / "_quality_reports"
META_DIR = SILVER_DIR / "_metadata"

TMP_EXTRACT_DIR = Path("/content/bathymetry_extract")

for d in [OUT_DIR, QC_DIR, META_DIR, TMP_EXTRACT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

BBOX_CANARIAS = {
    "lat_min": 27.0,
    "lat_max": 29.5,
    "lon_min": -18.5,
    "lon_max": -13.0,
}

# Margen suficiente para features hasta 2 km y búsqueda de isóbatas.
BBOX_MARGIN_DEG = 0.45

# Radio máximo para buscar isóbatas de 10/20 m.
ISOBATH_SEARCH_RADIUS_M = 30000

# Para depuración rápida. Deja None para todo.
MAX_ZONES_FOR_TEST = None

print("GEBCO_DIR existe:", GEBCO_DIR.exists())
print("EMODNET_DIR existe:", EMODNET_DIR.exists())
print("DIM_ZONE_PATH existe:", DIM_ZONE_PATH.exists())

if not DIM_ZONE_PATH.exists():
    raise FileNotFoundError("No existe beach_geography.parquet. Ejecuta primero 01_silver_dim_zone.ipynb.")

GEBCO_DIR existe: True
EMODNET_DIR existe: True
DIM_ZONE_PATH existe: True


## Celda 3 — Utilidades generales

In [4]:
def normalize_text(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    value = unicodedata.normalize("NFKD", value)
    value = "".join(c for c in value if not unicodedata.combining(c))
    value = re.sub(r"\s+", " ", value)
    return value.upper()


def normalize_col(col):
    col = normalize_text(col)
    if pd.isna(col):
        return ""
    col = re.sub(r"[^A-Z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col).strip("_")
    return col


def standardize_longitudes_to_180(lon_values):
    lon = np.asarray(lon_values)
    return ((lon + 180) % 360) - 180


def safe_float(value):
    try:
        if pd.isna(value):
            return np.nan
        return float(value)
    except Exception:
        return np.nan


def remove_if_exists(path):
    path = Path(path)
    if path.exists():
        if path.is_dir():
            shutil.rmtree(path)
        else:
            path.unlink()


GEOD = Geod(ellps="WGS84")


def geodesic_offset(lon, lat, bearing_deg, distance_m):
    """
    Devuelve lon/lat a distance_m desde un punto siguiendo bearing_deg.
    Bearing: 0=N, 90=E, 180=S, 270=W.
    """
    lon2, lat2, _ = GEOD.fwd(float(lon), float(lat), float(bearing_deg), float(distance_m))
    return lon2, lat2


def approximate_distance_m(lon0, lat0, lon_grid, lat_grid):
    """
    Distancia equirectangular aproximada en metros, suficiente para radios locales.
    """
    lat0_rad = np.deg2rad(lat0)
    dx = (lon_grid - lon0) * 111320.0 * np.cos(lat0_rad)
    dy = (lat_grid - lat0) * 110540.0
    return np.sqrt(dx ** 2 + dy ** 2)


def orientation_to_offshore_bearing(orientation):
    """
    Mapa simple de orientación/exposición costera a dirección offshore aproximada.
    Se usa para muestrear profundidad a 100m, 500m, 1km y 2km.
    """
    o = normalize_text(orientation)

    mapping = {
        "N": 0,
        "NE": 45,
        "E": 90,
        "SE": 135,
        "S": 180,
        "SW": 225,
        "W": 270,
        "NW": 315,
    }

    if pd.isna(o):
        return 0

    return mapping.get(o, 0)


def positive_depth_from_raw(values, sign_mode):
    """
    Convierte valores batimétricos a profundidad positiva.
    sign_mode:
    - negative_elevation_to_positive_depth: océano viene negativo.
    - positive_depth: profundidad ya viene positiva.
    """
    arr = np.asarray(values, dtype="float64")

    if sign_mode == "negative_elevation_to_positive_depth":
        depth = np.where(arr < 0, -arr, 0.0)
    else:
        depth = np.where(arr > 0, arr, 0.0)

    depth = np.where(np.isfinite(depth), depth, np.nan)

    return depth

## Celda 4 — Cargar `beach_geography`

In [5]:
beach_geography = pd.read_parquet(DIM_ZONE_PATH)

required_cols = ["zona_id", "nombre_zona", "isla", "municipio", "lat", "lon"]
missing = [c for c in required_cols if c not in beach_geography.columns]

if missing:
    raise ValueError(f"Faltan columnas en beach_geography: {missing}")

if MAX_ZONES_FOR_TEST is not None:
    beach_geography = beach_geography.head(MAX_ZONES_FOR_TEST).copy()

print("beach_geography shape:", beach_geography.shape)
print("zonas únicas:", beach_geography["zona_id"].nunique())

display(beach_geography.head())

beach_geography shape: (561, 17)
zonas únicas: 561


,zona_id,nombre_zona,isla,municipio,lat,lon,tipo_zona,orientacion_costa,exposicion_norte,exposicion_oeste,exposicion_este,exposicion_swell_nw,exposicion_swell_ne,vulnerabilidad_costera,vulnerabilidad_source,spatial_match_isla,spatial_match_municipio
0,CAN_TF_EL_PUERTITO_0,El Puertito,Tenerife,Güímar,28.2923,-16.3766,playa,NW,1,1,0,1,1,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
1,CAN_EH_LA_RESTINGA,La Restinga,El Hierro,El Pinar de El Hierro,27.6408,-17.9799,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
2,CAN_EH_ARENAS_BLANCAS,Arenas Blancas,El Hierro,Frontera,27.7667,-18.1218,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
3,CAN_EH_EL_VERODAL,El Verodal,El Hierro,Frontera,27.7471,-18.1512,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
4,CAN_EH_CHARCO_AZUL_0,Charco Azul,El Hierro,Frontera,27.7563,-18.0990,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True


## Celda 5 — Extraer ZIPs y localizar candidatos batimétricos

In [6]:
def extract_zip_if_needed(zip_path, extract_root):
    zip_path = Path(zip_path)
    extract_dir = extract_root / zip_path.stem

    if extract_dir.exists() and any(extract_dir.rglob("*")):
        return extract_dir

    extract_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_dir)

    return extract_dir


def collect_bathymetry_candidates():
    rows = []

    source_dirs = [
        ("GEBCO", GEBCO_DIR),
        ("EMODNET", EMODNET_DIR),
    ]

    for source_name, source_dir in source_dirs:
        if not source_dir.exists():
            continue

        direct_files = list(source_dir.rglob("*.nc")) + list(source_dir.rglob("*.tif")) + list(source_dir.rglob("*.tiff"))

        for p in direct_files:
            rows.append(
                {
                    "source": source_name,
                    "path": str(p),
                    "filename": p.name,
                    "file_type": p.suffix.lower().replace(".", ""),
                    "from_zip": False,
                    "size_mb": round(p.stat().st_size / 1024 / 1024, 2),
                }
            )

        for zp in source_dir.rglob("*.zip"):
            try:
                extract_dir = extract_zip_if_needed(zp, TMP_EXTRACT_DIR / source_name)
                inner_files = (
                    list(extract_dir.rglob("*.nc"))
                    + list(extract_dir.rglob("*.tif"))
                    + list(extract_dir.rglob("*.tiff"))
                )

                for p in inner_files:
                    rows.append(
                        {
                            "source": source_name,
                            "path": str(p),
                            "filename": p.name,
                            "file_type": p.suffix.lower().replace(".", ""),
                            "from_zip": True,
                            "zip_file": str(zp),
                            "size_mb": round(p.stat().st_size / 1024 / 1024, 2),
                        }
                    )
            except Exception as e:
                rows.append(
                    {
                        "source": source_name,
                        "path": str(zp),
                        "filename": zp.name,
                        "file_type": "zip_error",
                        "from_zip": True,
                        "zip_file": str(zp),
                        "error": repr(e),
                        "size_mb": round(zp.stat().st_size / 1024 / 1024, 2),
                    }
                )

    candidates = pd.DataFrame(rows)

    if candidates.empty:
        raise FileNotFoundError("No se encontraron archivos .nc/.tif/.tiff en GEBCO/EMODnet ni dentro de ZIPs.")

    # Prioridad: GEBCO NetCDF > EMODnet NetCDF > GEBCO GeoTIFF > EMODnet GeoTIFF.
    priority = []

    for _, r in candidates.iterrows():
        p = 99
        if r["source"] == "GEBCO" and r["file_type"] == "nc":
            p = 1
        elif r["source"] == "EMODNET" and r["file_type"] == "nc":
            p = 2
        elif r["source"] == "GEBCO" and r["file_type"] in ["tif", "tiff"]:
            p = 3
        elif r["source"] == "EMODNET" and r["file_type"] in ["tif", "tiff"]:
            p = 4
        priority.append(p)

    candidates["priority"] = priority
    candidates = candidates.sort_values(["priority", "size_mb"], ascending=[True, False]).reset_index(drop=True)

    return candidates


bathymetry_candidates = collect_bathymetry_candidates()

print("Candidatos batimétricos:")
display(bathymetry_candidates)

bathymetry_candidates.to_csv(META_DIR / "bathymetry_source_candidates.csv", index=False)

Candidatos batimétricos:


,source,path,filename,file_type,from_zip,zip_file,size_mb,priority
0,GEBCO,/content/bathymetry_extract/GEBCO/GEBCO_07_May...,gebco_2026_n30.8_s26.5_w-19.5_e-12.0.nc,nc,True,/content/drive/MyDrive/AI Projects/DeepWave Ca...,3.57,1
1,EMODNET,/content/bathymetry_extract/EMODNET/G3_2024.nc...,G3_2024.nc,nc,True,/content/drive/MyDrive/AI Projects/DeepWave Ca...,141.21,2
2,EMODNET,/content/bathymetry_extract/EMODNET/G3_2024.ti...,G3_2024.tif,tif,True,/content/drive/MyDrive/AI Projects/DeepWave Ca...,154.86,4


## Celda 6 — Funciones de apertura NetCDF/GeoTIFF

In [7]:
DEPTH_VAR_ALIASES = [
    "elevation",
    "ELEVATION",
    "z",
    "Z",
    "depth",
    "DEPTH",
    "bathymetry",
    "BATHYMETRY",
    "Band1",
    "band_data",
]


def find_coord_name(ds_in, candidates):
    for c in candidates:
        if c in ds_in.coords or c in ds_in.dims or c in ds_in.variables:
            return c
    return None


def find_bathymetry_var(ds_in):
    available = list(ds_in.data_vars)

    for alias in DEPTH_VAR_ALIASES:
        if alias in available:
            return alias

    norm_map = {normalize_col(v): v for v in available}

    for alias in DEPTH_VAR_ALIASES:
        a = normalize_col(alias)
        if a in norm_map:
            return norm_map[a]

    # Fallback: elegir variable 2D/3D con dimensiones lat/lon.
    for var in available:
        dims_norm = [normalize_col(d) for d in ds_in[var].dims]
        has_lat = any(d in ["LAT", "LATITUDE", "Y"] for d in dims_norm)
        has_lon = any(d in ["LON", "LONGITUDE", "X"] for d in dims_norm)

        if has_lat and has_lon:
            return var

    # Último fallback: primera variable numérica.
    for var in available:
        if np.issubdtype(ds_in[var].dtype, np.number):
            return var

    return None


def open_netcdf_bathymetry(path):
    last_error = None

    for engine in ["netcdf4", "h5netcdf", None]:
        try:
            if engine is None:
                ds_in = xr.open_dataset(path)
            else:
                ds_in = xr.open_dataset(path, engine=engine)
            break
        except Exception as e:
            last_error = e
            ds_in = None

    if ds_in is None:
        raise ValueError(f"No se pudo abrir NetCDF {Path(path).name}: {repr(last_error)}")

    lat_name = find_coord_name(ds_in, ["lat", "latitude", "y"])
    lon_name = find_coord_name(ds_in, ["lon", "longitude", "x"])

    if lat_name is None or lon_name is None:
        raise ValueError(f"{Path(path).name}: no se encontraron coordenadas lat/lon. Coords={list(ds_in.coords)}")

    rename_coords = {}

    if lat_name != "lat":
        rename_coords[lat_name] = "lat"

    if lon_name != "lon":
        rename_coords[lon_name] = "lon"

    if rename_coords:
        ds_in = ds_in.rename(rename_coords)

    # Convertir longitudes 0..360 a -180..180.
    lon_std = standardize_longitudes_to_180(ds_in["lon"].values)
    ds_in = ds_in.assign_coords(lon=lon_std)
    ds_in = ds_in.sortby("lon")
    ds_in = ds_in.sortby("lat")

    var = find_bathymetry_var(ds_in)

    if var is None:
        raise ValueError(f"{Path(path).name}: no se encontró variable batimétrica.")

    da = ds_in[var]

    # Eliminar dimensiones extra usando el primer índice.
    for d in list(da.dims):
        if d not in ["lat", "lon"]:
            print(f"AVISO {Path(path).name}: variable {var} tiene dimensión extra {d}; se usa índice 0.")
            da = da.isel({d: 0})

    da = da.sel(
        lat=slice(BBOX_CANARIAS["lat_min"] - BBOX_MARGIN_DEG, BBOX_CANARIAS["lat_max"] + BBOX_MARGIN_DEG),
        lon=slice(BBOX_CANARIAS["lon_min"] - BBOX_MARGIN_DEG, BBOX_CANARIAS["lon_max"] + BBOX_MARGIN_DEG),
    )

    if da.sizes.get("lat", 0) == 0 or da.sizes.get("lon", 0) == 0:
        raise ValueError(f"{Path(path).name}: recorte Canarias + margen vacío.")

    da = da.load()

    return da, {
        "open_method": "xarray_netcdf",
        "variable": var,
        "dims": dict(da.sizes),
    }


def open_geotiff_bathymetry(path):
    da = rioxarray.open_rasterio(path, masked=True)

    if "band" in da.dims:
        da = da.isel(band=0, drop=True)

    if da.rio.crs is not None and str(da.rio.crs).upper() not in ["EPSG:4326", "WGS84"]:
        # En principio los recortes descargados deberían ser manejables.
        da = da.rio.reproject("EPSG:4326")

    # Recorte espacial.
    da = da.rio.write_crs("EPSG:4326", inplace=False) if da.rio.crs is None else da
    da = da.rio.clip_box(
        minx=BBOX_CANARIAS["lon_min"] - BBOX_MARGIN_DEG,
        miny=BBOX_CANARIAS["lat_min"] - BBOX_MARGIN_DEG,
        maxx=BBOX_CANARIAS["lon_max"] + BBOX_MARGIN_DEG,
        maxy=BBOX_CANARIAS["lat_max"] + BBOX_MARGIN_DEG,
    )

    # Renombrar a lon/lat si viene como x/y.
    rename = {}
    if "x" in da.dims:
        rename["x"] = "lon"
    if "y" in da.dims:
        rename["y"] = "lat"

    if rename:
        da = da.rename(rename)

    da = da.sortby("lon")
    da = da.sortby("lat")
    da = da.load()

    return da, {
        "open_method": "rioxarray_geotiff",
        "variable": "raster_band_1",
        "dims": dict(da.sizes),
    }


def infer_sign_mode(da):
    values = np.asarray(da.values).astype("float64")
    finite = values[np.isfinite(values)]

    if len(finite) == 0:
        raise ValueError("La batimetría no contiene valores finitos en Canarias.")

    q05 = np.nanpercentile(finite, 5)
    q50 = np.nanpercentile(finite, 50)
    q95 = np.nanpercentile(finite, 95)

    # GEBCO/EMODnet normalmente dan elevación negativa bajo el mar.
    if q50 < 0 or q05 < -1:
        return "negative_elevation_to_positive_depth", {"q05": q05, "q50": q50, "q95": q95}

    return "positive_depth", {"q05": q05, "q50": q50, "q95": q95}


def load_best_bathymetry(candidates):
    errors = []

    for _, row in candidates.iterrows():
        path = Path(row["path"])
        file_type = row["file_type"]

        if not path.exists():
            continue

        try:
            if file_type == "nc":
                da, meta = open_netcdf_bathymetry(path)
            elif file_type in ["tif", "tiff"]:
                da, meta = open_geotiff_bathymetry(path)
            else:
                continue

            sign_mode, sign_stats = infer_sign_mode(da)

            meta.update(
                {
                    "source": row["source"],
                    "path": str(path),
                    "filename": path.name,
                    "file_type": file_type,
                    "sign_mode": sign_mode,
                    "sign_stats": sign_stats,
                }
            )

            return da, meta, pd.DataFrame(errors)

        except Exception as e:
            errors.append(
                {
                    "source": row.get("source"),
                    "path": str(path),
                    "filename": path.name,
                    "file_type": file_type,
                    "error": repr(e),
                }
            )

    raise ValueError("No se pudo abrir ningún candidato batimétrico. Revisar quality_bathymetry_source_errors.csv.")

## Celda 7 — Cargar mejor fuente batimétrica disponible

In [8]:
bathy_da, bathy_meta, bathy_source_errors = load_best_bathymetry(bathymetry_candidates)

print("Fuente batimétrica seleccionada:")
print(json.dumps(bathy_meta, indent=2, ensure_ascii=False, default=str))

print("Errores previos de fuentes descartadas:")
display(bathy_source_errors)

bathy_source_errors.to_csv(QC_DIR / "quality_bathymetry_source_errors.csv", index=False)

with open(META_DIR / "bathymetry_selected_source.json", "w", encoding="utf-8") as f:
    json.dump(bathy_meta, f, indent=2, ensure_ascii=False, default=str)

print("DataArray:")
print(bathy_da)

Fuente batimétrica seleccionada:
{
  "open_method": "xarray_netcdf",
  "variable": "elevation",
  "dims": {
    "lat": 816,
    "lon": 1536
  },
  "source": "GEBCO",
  "path": "/content/bathymetry_extract/GEBCO/GEBCO_07_May_2026_45ce6b3f4717/gebco_2026_n30.8_s26.5_w-19.5_e-12.0.nc",
  "filename": "gebco_2026_n30.8_s26.5_w-19.5_e-12.0.nc",
  "file_type": "nc",
  "sign_mode": "negative_elevation_to_positive_depth",
  "sign_stats": {
    "q05": -4239.0,
    "q50": -3185.0,
    "q95": 89.0
  }
}
Errores previos de fuentes descartadas:


""


DataArray:
<xarray.DataArray 'elevation' (lat: 816, lon: 1536)> Size: 5MB
array([[-3649., -3651., -3652., ...,   207.,   207.,   208.],
       [-3651., -3652., -3653., ...,   206.,   207.,   210.],
       [-3651., -3653., -3654., ...,   206.,   208.,   211.],
       ...,
       [-4624., -4622., -4620., ...,  -306.,  -315.,  -329.],
       [-4622., -4619., -4617., ...,  -303.,  -311.,  -328.],
       [-4623., -4620., -4618., ...,  -303.,  -311.,  -328.]],
      dtype=float32)
Coordinates:
  * lat      (lat) float64 7kB 26.55 26.56 26.56 26.56 ... 29.94 29.94 29.95
  * lon      (lon) float64 12kB -18.95 -18.94 -18.94 ... -12.56 -12.56 -12.55
Attributes:
    long_name:           Elevation relative to sea level
    grid_mapping:        crs
    sdn_parameter_name:  Sea floor height (above mean sea level) {bathymetric...
    sdn_parameter_urn:   SDN:P01::BATHHGHT
    sdn_uom_name:        Metres
    sdn_uom_urn:         SDN:P06::ULAA
    standard_name:       height_above_mean_sea_level
    un

## Celda 8 — Funciones de muestreo y cálculo de features

In [9]:
def sample_depth_at(lon, lat):
    """
    Muestrea profundidad positiva en el punto más cercano.
    """
    try:
        raw = bathy_da.interp(lon=float(lon), lat=float(lat), method="nearest").values
        raw = float(np.asarray(raw))
        depth = positive_depth_from_raw(np.array([raw]), bathy_meta["sign_mode"])[0]
        return float(depth) if np.isfinite(depth) else np.nan
    except Exception:
        return np.nan


def get_depth_patch(lon0, lat0, radius_m):
    """
    Devuelve parche local de profundidad positiva y distancias al punto.
    """
    lon0 = float(lon0)
    lat0 = float(lat0)

    lat_radius = radius_m / 110540.0
    lon_radius = radius_m / (111320.0 * max(np.cos(np.deg2rad(lat0)), 0.1))

    patch = bathy_da.sel(
        lat=slice(lat0 - lat_radius, lat0 + lat_radius),
        lon=slice(lon0 - lon_radius, lon0 + lon_radius),
    )

    if patch.sizes.get("lat", 0) == 0 or patch.sizes.get("lon", 0) == 0:
        return None

    raw_values = np.asarray(patch.values).astype("float64")
    depth = positive_depth_from_raw(raw_values, bathy_meta["sign_mode"])

    lats = np.asarray(patch["lat"].values).astype("float64")
    lons = np.asarray(patch["lon"].values).astype("float64")

    lon_grid, lat_grid = np.meshgrid(lons, lats)
    dist = approximate_distance_m(lon0, lat0, lon_grid, lat_grid)

    return {
        "depth": depth,
        "distance_m": dist,
        "lat_grid": lat_grid,
        "lon_grid": lon_grid,
    }


def mean_depth_within_radius(lon, lat, radius_m):
    patch = get_depth_patch(lon, lat, radius_m)

    if patch is None:
        return np.nan

    depth = patch["depth"]
    dist = patch["distance_m"]

    mask = (dist <= radius_m) & np.isfinite(depth) & (depth > 0)

    if not np.any(mask):
        return np.nan

    return float(np.nanmean(depth[mask]))


def bathymetry_roughness_within_radius(lon, lat, radius_m=1000):
    patch = get_depth_patch(lon, lat, radius_m)

    if patch is None:
        return np.nan

    depth = patch["depth"]
    dist = patch["distance_m"]

    mask = (dist <= radius_m) & np.isfinite(depth) & (depth > 0)

    if np.sum(mask) < 5:
        return np.nan

    return float(np.nanstd(depth[mask]))


def distance_to_isobath(lon, lat, target_depth_m, search_radius_m=ISOBATH_SEARCH_RADIUS_M):
    patch = get_depth_patch(lon, lat, search_radius_m)

    if patch is None:
        return np.nan

    depth = patch["depth"]
    dist = patch["distance_m"]

    mask = (dist <= search_radius_m) & np.isfinite(depth) & (depth > 0)

    if np.sum(mask) < 5:
        return np.nan

    diff = np.abs(depth - target_depth_m)
    diff_masked = np.where(mask, diff, np.nan)

    min_diff = np.nanmin(diff_masked)

    # Tolerancia flexible para evitar no encontrar la isóbata por resolución de malla.
    tolerance = max(2.5, target_depth_m * 0.25)

    if not np.isfinite(min_diff) or min_diff > tolerance:
        return np.nan

    nearest_mask = mask & np.isfinite(diff) & (diff == min_diff)

    return float(np.nanmin(dist[nearest_mask]))


def compute_features_for_zone(row):
    lon = float(row["lon"])
    lat = float(row["lat"])

    orientation = row.get("orientacion_costa", None)
    bearing = orientation_to_offshore_bearing(orientation)

    # Profundidades a puntos offshore aproximados.
    offshore_points = {}

    for d in [100, 500, 1000, 2000]:
        lon_d, lat_d = geodesic_offset(lon, lat, bearing, d)
        offshore_points[d] = (lon_d, lat_d)

    depth_100m = sample_depth_at(*offshore_points[100])
    depth_500m = sample_depth_at(*offshore_points[500])
    depth_1km = sample_depth_at(*offshore_points[1000])
    depth_2km = sample_depth_at(*offshore_points[2000])

    mean_depth_1km = mean_depth_within_radius(lon, lat, 1000)
    roughness_1km = bathymetry_roughness_within_radius(lon, lat, 1000)

    # Pendientes positivas hacia mar abierto.
    slope_0_500m = np.nan
    slope_500m_2km = np.nan

    if pd.notna(depth_100m) and pd.notna(depth_500m):
        slope_0_500m = (depth_500m - depth_100m) / 400.0

    if pd.notna(depth_500m) and pd.notna(depth_2km):
        slope_500m_2km = (depth_2km - depth_500m) / 1500.0

    dist_10m = distance_to_isobath(lon, lat, 10)
    dist_20m = distance_to_isobath(lon, lat, 20)

    return {
        "offshore_bearing_deg": bearing,
        "depth_100m": depth_100m,
        "depth_500m": depth_500m,
        "depth_1km": depth_1km,
        "depth_2km": depth_2km,
        "mean_depth_1km": mean_depth_1km,
        "slope_0_500m": slope_0_500m,
        "slope_500m_2km": slope_500m_2km,
        "distance_to_10m_isobath": dist_10m,
        "distance_to_20m_isobath": dist_20m,
        "bathymetry_roughness": roughness_1km,
    }

## Celda 9 — Calcular features para todas las zonas

In [10]:
feature_rows = []
feature_errors = []

for _, row in tqdm(beach_geography.iterrows(), total=len(beach_geography), desc="Calculando bathymetry features"):
    try:
        features = compute_features_for_zone(row)

        base = {
            "zona_id": row["zona_id"],
            "nombre_zona": row.get("nombre_zona"),
            "isla": row.get("isla"),
            "municipio": row.get("municipio"),
            "lat": row.get("lat"),
            "lon": row.get("lon"),
            "orientacion_costa": row.get("orientacion_costa", np.nan),
            "source": bathy_meta["source"],
            "bathymetry_source": bathy_meta["source"],
            "bathymetry_file": bathy_meta["filename"],
            "bathymetry_variable": bathy_meta["variable"],
            "bathymetry_open_method": bathy_meta["open_method"],
            "bathymetry_sign_mode": bathy_meta["sign_mode"],
            "feature_method": "offshore_bearing_and_local_radius",
            "isobath_search_radius_m": ISOBATH_SEARCH_RADIUS_M,
        }

        base.update(features)
        feature_rows.append(base)

    except Exception as e:
        feature_errors.append(
            {
                "zona_id": row.get("zona_id"),
                "nombre_zona": row.get("nombre_zona"),
                "error": repr(e),
            }
        )

bathymetry_features = pd.DataFrame(feature_rows)
feature_errors_df = pd.DataFrame(feature_errors)

print("Features calculadas:", len(bathymetry_features))
print("Errores:", len(feature_errors_df))

display(bathymetry_features.head())
display(feature_errors_df)

feature_errors_df.to_csv(QC_DIR / "quality_bathymetry_feature_errors.csv", index=False)

if len(feature_errors_df):
    print("AVISO: algunas zonas no pudieron calcularse. Se documentan en quality_bathymetry_feature_errors.csv.")

Calculando bathymetry features:   0%|          | 0/561 [00:00<?, ?it/s]

Features calculadas: 561
Errores: 0


,zona_id,nombre_zona,isla,municipio,lat,lon,orientacion_costa,source,bathymetry_source,bathymetry_file,...,depth_100m,depth_500m,depth_1km,depth_2km,mean_depth_1km,slope_0_500m,slope_500m_2km,distance_to_10m_isobath,distance_to_20m_isobath,bathymetry_roughness
0,CAN_TF_EL_PUERTITO_0,El Puertito,Tenerife,Güímar,28.2923,-16.3766,NW,GEBCO,GEBCO,gebco_2026_n30.8_s26.5_w-19.5_e-12.0.nc,...,0.0,0.0,0.0,0.0,22.750000,0.000,0.000000,1188.750957,10147.462370,NaN
1,CAN_EH_LA_RESTINGA,La Restinga,El Hierro,El Pinar de El Hierro,27.6408,-17.9799,W,GEBCO,GEBCO,gebco_2026_n30.8_s26.5_w-19.5_e-12.0.nc,...,2.0,2.0,50.0,272.0,40.222222,0.000,0.180000,4036.751910,1091.588795,36.753769
2,CAN_EH_ARENAS_BLANCAS,Arenas Blancas,El Hierro,Frontera,27.7667,-18.1218,W,GEBCO,GEBCO,gebco_2026_n30.8_s26.5_w-19.5_e-12.0.nc,...,30.0,34.0,7.0,127.0,158.181818,0.010,0.062000,13592.374725,1918.453860,160.406199
3,CAN_EH_EL_VERODAL,El Verodal,El Hierro,Frontera,27.7471,-18.1512,W,GEBCO,GEBCO,gebco_2026_n30.8_s26.5_w-19.5_e-12.0.nc,...,17.0,47.0,74.0,560.0,40.000000,0.075,0.342000,16873.135810,1644.860913,23.151674
4,CAN_EH_CHARCO_AZUL_0,Charco Azul,El Hierro,Frontera,27.7563,-18.0990,W,GEBCO,GEBCO,gebco_2026_n30.8_s26.5_w-19.5_e-12.0.nc,...,32.0,32.0,0.0,0.0,114.500000,0.000,-0.021333,11642.928756,714.227034,115.856592


""


## Celda 10 — Flags de calidad y validación

In [11]:
FEATURE_COLUMNS = [
    "depth_100m",
    "depth_500m",
    "depth_1km",
    "depth_2km",
    "mean_depth_1km",
    "slope_0_500m",
    "slope_500m_2km",
    "distance_to_10m_isobath",
    "distance_to_20m_isobath",
    "bathymetry_roughness",
]


def add_feature_flags(df):
    df = df.copy()

    for col in FEATURE_COLUMNS:
        if col not in df.columns:
            df[col] = np.nan

        flag_col = f"{col}_flag"
        df[flag_col] = 0

        missing = df[col].isna()
        df.loc[missing, flag_col] = 1

        # Rango físico amplio.
        if col.startswith("depth") or col.startswith("mean_depth"):
            outlier = (~missing) & ((df[col] < 0) | (df[col] > 7000))
        elif col.startswith("distance"):
            outlier = (~missing) & ((df[col] < 0) | (df[col] > ISOBATH_SEARCH_RADIUS_M))
        elif col.startswith("slope"):
            outlier = (~missing) & (df[col].abs() > 5)
        else:
            outlier = (~missing) & ((df[col] < 0) | (df[col] > 7000))

        df.loc[outlier, flag_col] = 2
        df[flag_col] = df[flag_col].astype("int8")

    return df


bathymetry_features = add_feature_flags(bathymetry_features)

if bathymetry_features.empty:
    raise ValueError("bathymetry_features está vacío.")

if bathymetry_features["zona_id"].duplicated().any():
    duplicated = bathymetry_features[bathymetry_features["zona_id"].duplicated(keep=False)]
    display(duplicated)
    raise ValueError("Hay zona_id duplicados en bathymetry_features.")

missing_core = bathymetry_features[["zona_id", "lat", "lon"]].isna().sum()

if missing_core.any():
    display(missing_core)
    raise ValueError("Hay nulos en zona_id/lat/lon.")

# Avisos, no bloqueo: isóbatas pueden no encontrarse en zonas muy abruptas o por resolución.
for col in FEATURE_COLUMNS:
    pct = bathymetry_features[col].isna().mean() * 100
    print(f"{col}: missing {pct:.2f}%")

print("Validación básica bathymetry_features superada.")
display(bathymetry_features.head())

depth_100m: missing 0.00%
depth_500m: missing 0.00%
depth_1km: missing 0.00%
depth_2km: missing 0.00%
mean_depth_1km: missing 0.71%
slope_0_500m: missing 0.00%
slope_500m_2km: missing 0.00%
distance_to_10m_isobath: missing 0.00%
distance_to_20m_isobath: missing 0.00%
bathymetry_roughness: missing 17.65%
Validación básica bathymetry_features superada.


,zona_id,nombre_zona,isla,municipio,lat,lon,orientacion_costa,source,bathymetry_source,bathymetry_file,...,depth_100m_flag,depth_500m_flag,depth_1km_flag,depth_2km_flag,mean_depth_1km_flag,slope_0_500m_flag,slope_500m_2km_flag,distance_to_10m_isobath_flag,distance_to_20m_isobath_flag,bathymetry_roughness_flag
0,CAN_TF_EL_PUERTITO_0,El Puertito,Tenerife,Güímar,28.2923,-16.3766,NW,GEBCO,GEBCO,gebco_2026_n30.8_s26.5_w-19.5_e-12.0.nc,...,0,0,0,0,0,0,0,0,0,1
1,CAN_EH_LA_RESTINGA,La Restinga,El Hierro,El Pinar de El Hierro,27.6408,-17.9799,W,GEBCO,GEBCO,gebco_2026_n30.8_s26.5_w-19.5_e-12.0.nc,...,0,0,0,0,0,0,0,0,0,0
2,CAN_EH_ARENAS_BLANCAS,Arenas Blancas,El Hierro,Frontera,27.7667,-18.1218,W,GEBCO,GEBCO,gebco_2026_n30.8_s26.5_w-19.5_e-12.0.nc,...,0,0,0,0,0,0,0,0,0,0
3,CAN_EH_EL_VERODAL,El Verodal,El Hierro,Frontera,27.7471,-18.1512,W,GEBCO,GEBCO,gebco_2026_n30.8_s26.5_w-19.5_e-12.0.nc,...,0,0,0,0,0,0,0,0,0,0
4,CAN_EH_CHARCO_AZUL_0,Charco Azul,El Hierro,Frontera,27.7563,-18.0990,W,GEBCO,GEBCO,gebco_2026_n30.8_s26.5_w-19.5_e-12.0.nc,...,0,0,0,0,0,0,0,0,0,0


## Celda 11 — Reportes de calidad

In [12]:
quality_summary = pd.DataFrame(
    [
        {
            "table": "bathymetry_features",
            "source": bathy_meta["source"],
            "rows": len(bathymetry_features),
            "unique_zona_id": bathymetry_features["zona_id"].nunique(),
            "bathymetry_file": bathy_meta["filename"],
            "bathymetry_variable": bathy_meta["variable"],
            "bathymetry_sign_mode": bathy_meta["sign_mode"],
            "bbox_margin_deg": BBOX_MARGIN_DEG,
            "isobath_search_radius_m": ISOBATH_SEARCH_RADIUS_M,
            **{f"{col}_missing_pct": float(bathymetry_features[col].isna().mean() * 100) for col in FEATURE_COLUMNS},
        }
    ]
)

missing_by_column = (
    bathymetry_features.isna()
    .mean()
    .mul(100)
    .reset_index()
    .rename(columns={"index": "column", 0: "missing_pct"})
)

feature_stats = (
    bathymetry_features[FEATURE_COLUMNS]
    .describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])
    .T
    .reset_index()
    .rename(columns={"index": "feature"})
)

display(quality_summary)
display(missing_by_column)
display(feature_stats)

quality_summary.to_csv(QC_DIR / "quality_bathymetry_features_summary.csv", index=False)
missing_by_column.to_csv(QC_DIR / "quality_bathymetry_features_missing_by_column.csv", index=False)
feature_stats.to_csv(QC_DIR / "quality_bathymetry_features_stats.csv", index=False)

,table,source,rows,unique_zona_id,bathymetry_file,bathymetry_variable,bathymetry_sign_mode,bbox_margin_deg,isobath_search_radius_m,depth_100m_missing_pct,depth_500m_missing_pct,depth_1km_missing_pct,depth_2km_missing_pct,mean_depth_1km_missing_pct,slope_0_500m_missing_pct,slope_500m_2km_missing_pct,distance_to_10m_isobath_missing_pct,distance_to_20m_isobath_missing_pct,bathymetry_roughness_missing_pct
0,bathymetry_features,GEBCO,561,561,gebco_2026_n30.8_s26.5_w-19.5_e-12.0.nc,elevation,negative_elevation_to_positive_depth,0.45,30000,0.0,0.0,0.0,0.0,0.713012,0.0,0.0,0.0,0.0,17.647059


,column,missing_pct
0,zona_id,0.000000
1,nombre_zona,0.000000
2,isla,0.000000
3,municipio,0.000000
4,lat,0.000000
5,lon,0.000000
6,orientacion_costa,0.000000
7,source,0.000000
8,bathymetry_source,0.000000
9,bathymetry_file,0.000000


,feature,count,mean,std,min,5%,25%,50%,75%,95%,max
0,depth_100m,561.0,6.117647,13.051289,0.000000,0.000000,0.000000,0.000000,6.000000,30.000000,131.000000
1,depth_500m,561.0,9.016043,19.768068,0.000000,0.000000,0.000000,0.000000,10.000000,40.000000,197.000000
2,depth_1km,561.0,16.320856,36.509594,0.000000,0.000000,0.000000,0.000000,17.000000,82.000000,332.000000
3,depth_2km,561.0,40.951872,94.449511,0.000000,0.000000,0.000000,1.000000,34.000000,256.000000,815.000000
4,mean_depth_1km,557.0,24.978435,29.930736,1.000000,4.000000,8.923077,16.333333,26.500000,88.450000,202.400000
5,slope_0_500m,561.0,0.007246,0.032041,-0.177500,-0.017500,0.000000,0.000000,0.007500,0.055000,0.275000
6,slope_500m_2km,561.0,0.021291,0.055210,-0.079333,-0.003333,0.000000,0.000000,0.017333,0.150000,0.412000
7,distance_to_10m_isobath,561.0,4856.002048,4980.761540,108.370998,514.020414,1264.143732,2836.098722,6939.240014,16721.865614,24203.330243
8,distance_to_20m_isobath,561.0,3057.764736,2509.596348,130.607321,586.982244,1330.020381,2328.469407,3759.266232,9027.517545,12139.081313
9,bathymetry_roughness,462.0,16.737578,26.343710,0.489898,2.332713,4.583083,7.588577,14.352623,69.669806,194.380034


## Celda 12 — Guardar Parquet

In [13]:
OUT_PARQUET = OUT_DIR / "bathymetry_features.parquet"

final_cols = [
    "zona_id",
    "nombre_zona",
    "isla",
    "municipio",
    "lat",
    "lon",
    "orientacion_costa",
    "offshore_bearing_deg",
    "depth_100m",
    "depth_500m",
    "depth_1km",
    "depth_2km",
    "mean_depth_1km",
    "slope_0_500m",
    "slope_500m_2km",
    "distance_to_10m_isobath",
    "distance_to_20m_isobath",
    "bathymetry_roughness",
    "source",
    "bathymetry_source",
    "bathymetry_file",
    "bathymetry_variable",
    "bathymetry_open_method",
    "bathymetry_sign_mode",
    "feature_method",
    "isobath_search_radius_m",
]

flag_cols = [c for c in bathymetry_features.columns if c.endswith("_flag")]
final_cols = final_cols + flag_cols

for col in final_cols:
    if col not in bathymetry_features.columns:
        bathymetry_features[col] = np.nan

bathymetry_final = bathymetry_features[final_cols].copy()

remove_if_exists(OUT_PARQUET)

bathymetry_final.to_parquet(
    OUT_PARQUET,
    index=False,
    engine="pyarrow",
    compression="snappy",
)

print("Guardado:")
print(OUT_PARQUET)
print("Shape:", bathymetry_final.shape)
display(bathymetry_final.head())

Guardado:
/content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/bathymetry_features/bathymetry_features.parquet
Shape: (561, 36)


,zona_id,nombre_zona,isla,municipio,lat,lon,orientacion_costa,offshore_bearing_deg,depth_100m,depth_500m,...,depth_100m_flag,depth_500m_flag,depth_1km_flag,depth_2km_flag,mean_depth_1km_flag,slope_0_500m_flag,slope_500m_2km_flag,distance_to_10m_isobath_flag,distance_to_20m_isobath_flag,bathymetry_roughness_flag
0,CAN_TF_EL_PUERTITO_0,El Puertito,Tenerife,Güímar,28.2923,-16.3766,NW,315,0.0,0.0,...,0,0,0,0,0,0,0,0,0,1
1,CAN_EH_LA_RESTINGA,La Restinga,El Hierro,El Pinar de El Hierro,27.6408,-17.9799,W,270,2.0,2.0,...,0,0,0,0,0,0,0,0,0,0
2,CAN_EH_ARENAS_BLANCAS,Arenas Blancas,El Hierro,Frontera,27.7667,-18.1218,W,270,30.0,34.0,...,0,0,0,0,0,0,0,0,0,0
3,CAN_EH_EL_VERODAL,El Verodal,El Hierro,Frontera,27.7471,-18.1512,W,270,17.0,47.0,...,0,0,0,0,0,0,0,0,0,0
4,CAN_EH_CHARCO_AZUL_0,Charco Azul,El Hierro,Frontera,27.7563,-18.0990,W,270,32.0,32.0,...,0,0,0,0,0,0,0,0,0,0


## Celda 13 — Comprobación final de lectura

In [14]:
test = pd.read_parquet(OUT_PARQUET)

print("Parquet leído correctamente.")
print("Shape:", test.shape)
print("Zonas únicas:", test["zona_id"].nunique())

display(test.head())

if test.empty:
    raise ValueError("El parquet final está vacío.")

if test["zona_id"].duplicated().any():
    raise ValueError("El parquet final tiene zona_id duplicados.")

# Validación final flexible.
essential_missing = test[["depth_500m", "depth_1km", "mean_depth_1km"]].isna().mean()

display(
    essential_missing
    .mul(100)
    .reset_index()
    .rename(columns={"index": "feature", 0: "missing_pct"})
)

if essential_missing["mean_depth_1km"] > 0.5:
    print(
        "AVISO: más del 50% de mean_depth_1km está nulo. "
        "Puede indicar que la fuente batimétrica no cubre bien la costa o que la resolución es baja."
    )

print("Reportes de calidad bathymetry:")
for p in sorted(QC_DIR.glob("quality_bathymetry*.csv")):
    print("-", p)

print("\nMetadatos bathymetry:")
for p in sorted(META_DIR.glob("bathymetry*.csv")):
    print("-", p)

print("\nValidación final bathymetry_features superada.")

Parquet leído correctamente.
Shape: (561, 36)
Zonas únicas: 561


,zona_id,nombre_zona,isla,municipio,lat,lon,orientacion_costa,offshore_bearing_deg,depth_100m,depth_500m,...,depth_100m_flag,depth_500m_flag,depth_1km_flag,depth_2km_flag,mean_depth_1km_flag,slope_0_500m_flag,slope_500m_2km_flag,distance_to_10m_isobath_flag,distance_to_20m_isobath_flag,bathymetry_roughness_flag
0,CAN_TF_EL_PUERTITO_0,El Puertito,Tenerife,Güímar,28.2923,-16.3766,NW,315,0.0,0.0,...,0,0,0,0,0,0,0,0,0,1
1,CAN_EH_LA_RESTINGA,La Restinga,El Hierro,El Pinar de El Hierro,27.6408,-17.9799,W,270,2.0,2.0,...,0,0,0,0,0,0,0,0,0,0
2,CAN_EH_ARENAS_BLANCAS,Arenas Blancas,El Hierro,Frontera,27.7667,-18.1218,W,270,30.0,34.0,...,0,0,0,0,0,0,0,0,0,0
3,CAN_EH_EL_VERODAL,El Verodal,El Hierro,Frontera,27.7471,-18.1512,W,270,17.0,47.0,...,0,0,0,0,0,0,0,0,0,0
4,CAN_EH_CHARCO_AZUL_0,Charco Azul,El Hierro,Frontera,27.7563,-18.0990,W,270,32.0,32.0,...,0,0,0,0,0,0,0,0,0,0


,feature,missing_pct
0,depth_500m,0.000000
1,depth_1km,0.000000
2,mean_depth_1km,0.713012


Reportes de calidad bathymetry:
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_bathymetry_feature_errors.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_bathymetry_features_missing_by_column.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_bathymetry_features_stats.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_bathymetry_features_summary.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_bathymetry_source_errors.csv

Metadatos bathymetry:
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_metadata/bathymetry_source_candidates.csv

Validación final bathymetry_features superada.


## Resultado esperado

Al terminar deberían existir:

```text
silver/bathymetry_features/bathymetry_features.parquet
silver/_quality_reports/quality_bathymetry_features_summary.csv
silver/_quality_reports/quality_bathymetry_features_missing_by_column.csv
silver/_quality_reports/quality_bathymetry_features_stats.csv
silver/_metadata/bathymetry_selected_source.json
silver/_metadata/bathymetry_source_candidates.csv
```

Comprobaciones principales:

```text
Features calculadas = número de zonas de beach_geography
Parquet leído correctamente
zona_id únicos = filas
Validación final bathymetry_features superada
```

Las distancias a isóbatas pueden quedar `NaN` en zonas donde la resolución o la búsqueda local no encuentre una isóbata de 10/20 m dentro del radio configurado.